# NoiseBridge — Kaggle runner

**Before running:** in the right sidebar, attach your `hinglish-noisy` dataset (or whichever Kaggle dataset has `train.csv`, `val.csv`, `test_clean.csv`, `test_noisy_low.csv`, `test_noisy_medium.csv`, `test_noisy_high.csv`) and note the path — usually `/kaggle/input/hinglish-noisy/`.

**Edit the CONFIG cell below** for each account/session — change `ENCODER` and `SEED`. Everything else stays the same.

## 1. Config — edit these per account

In [ ]:
# ─── EDIT PER ACCOUNT ───
ENCODER = 'xlm-roberta-base'         # or 'bert-base-multilingual-cased' or 'ai4bharat/indic-bert'
SEED    = 42                          # 42 / 43 / 44

# ─── Should not need to change ───
DATASET_DIR = '/kaggle/input/hinglish-noisy'   # path to the Kaggle dataset that contains train.csv etc
EPOCHS      = 5
BATCH_SIZE  = 16                       # XLM-R / mBERT / MuRIL handle 16 on P100/T4 with fp16
LR          = 3e-5
ALPHA       = 0.15                     # PWNIC weight
GAMMA       = 0.2                      # aux noise-prediction weight
MAX_LEN     = 128                      # transformer encoders
FP16        = True
NUM_WORKERS = 2

## 2. Install deps and verify GPU

In [ ]:
!pip install -q jellyfish
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

## 3. Stage data and code into the working directory

The trainer expects data in `./data/final/` and the two .py files alongside the notebook.
Upload `noisebridge.py` and `train_noisebridge.py` as a Kaggle dataset (or paste them via the next cell), then run this.

In [ ]:
import os, shutil

os.makedirs('data/final', exist_ok=True)
for fname in ['train.csv', 'val.csv', 'test_clean.csv',
              'test_noisy_low.csv', 'test_noisy_medium.csv', 'test_noisy_high.csv']:
    src = os.path.join(DATASET_DIR, fname)
    dst = os.path.join('data/final', fname)
    if not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f'  staged {fname}')
    else:
        print(f'  exists  {fname}')

print('\nData ready:')
!ls -la data/final/

## 4. Drop in `noisebridge.py` and `train_noisebridge.py`

Easiest path: upload them as a Kaggle dataset called e.g. `noisebridge-code` and uncomment the copy cell.  
Alternative: paste their contents into two `%%writefile` cells (uglier but works in a pinch).

In [ ]:
# Option A: code attached as a Kaggle dataset
CODE_DIR = '/kaggle/input/noisebridge-code'
if os.path.exists(CODE_DIR):
    for f in ['noisebridge.py', 'train_noisebridge.py']:
        shutil.copy(os.path.join(CODE_DIR, f), f)
    print('  copied code from dataset')

assert os.path.exists('noisebridge.py'),       'noisebridge.py is missing'
assert os.path.exists('train_noisebridge.py'), 'train_noisebridge.py is missing'
print('  code ready')

## 5. Run NoiseBridge — one unified training, eval on all 4 test sets

In [ ]:
import subprocess, sys, time

cmd = [
    sys.executable, 'train_noisebridge.py',
    '--encoder',    ENCODER,
    '--noise',      'all',
    '--seeds',      str(SEED),
    '--epochs',     str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--lr',         str(LR),
    '--alpha',      str(ALPHA),
    '--gamma',      str(GAMMA),
    '--max_len',    str(MAX_LEN),
    '--num_workers',str(NUM_WORKERS),
]
if FP16:
    cmd.append('--fp16')

print('Command:', ' '.join(cmd))
print()

t0 = time.time()
# Stream output live so you can watch the per-epoch progress
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'\nElapsed: {(time.time() - t0)/60:.1f} min')

## 6. Show results JSONs and copy them to `/kaggle/working` so they're downloadable

In [ ]:
import json, glob, pandas as pd

files = sorted(glob.glob('results/tables/*.json'))
rows = []
for f in files:
    with open(f) as fp:
        rows.append(json.load(fp))

df = pd.DataFrame(rows)
cols = ['encoder', 'train_noise', 'test_noise', 'seed', 'best_val_f1', 'test_f1_macro', 'enable_aux']
print(df[cols].to_string(index=False))

# Copy result jsons to /kaggle/working so they're saved with the notebook output
os.makedirs('/kaggle/working/results', exist_ok=True)
for f in files:
    shutil.copy(f, '/kaggle/working/results/')
print(f'\nCopied {len(files)} result files to /kaggle/working/results/')